Imports and Setup

In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


*Load the Datasets*

Ensure your files are in the same directory, or adjust the path if you are using Kaggle's

In [12]:
bbb_df = pd.read_csv('train_IPL.csv')
public_lb = pd.read_csv('public_lb_matches.csv')
schedule = pd.read_csv('schedule.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Training data shape: {bbb_df.shape}")

Training data shape: (272704, 38)


Entity Consolidation (Cleaning Names)

In [13]:
team_mapping = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

def clean_team_names(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].replace(team_mapping)
    return df

# Clean the ball-by-ball data immediately
bbb_df = clean_team_names(bbb_df, ['Bat First', 'Bat Second', 'toss_winner', 'match_won_by'])

Data Aggregation & Target Derivation

In [14]:
# Group to match level
match_df = bbb_df.groupby('Match ID').agg({
    'Bat First': 'first',
    'Bat Second': 'first',
    'Venue': 'first',
    'toss_winner': 'first',
    'toss_decision': 'first',
    'match_won_by': 'first'
}).reset_index()

# Calculate Innings Totals and Wickets
innings_stats = bbb_df.groupby(['Match ID', 'Innings']).agg({
    'Runs From Ball': 'sum',
    'Extra Runs': 'sum',
    'Wicket': 'sum'
}).reset_index()
innings_stats['Total Runs'] = innings_stats['Runs From Ball'] + innings_stats['Extra Runs']

targets = []
for _, row in match_df.iterrows():
    match_id = row['Match ID']
    team_a = row['Bat First']
    team_b = row['Bat Second']
    winner = row['match_won_by']
    
    match_innings = innings_stats[innings_stats['Match ID'] == match_id]
    
    try:
        runs_A = match_innings[match_innings['Innings'] == 1]['Total Runs'].values[0]
        wkts_A = match_innings[match_innings['Innings'] == 1]['Wicket'].values[0]
        runs_B = match_innings[match_innings['Innings'] == 2]['Total Runs'].values[0]
        wkts_B = match_innings[match_innings['Innings'] == 2]['Wicket'].values[0]
    except IndexError:
        targets.append(np.nan)
        continue

    if winner == team_a:
        margin_runs = runs_A - runs_B
        targets.append('A_big' if margin_runs > 20 else 'A_small')
    elif winner == team_b:
        margin_wickets = 10 - wkts_B
        targets.append('B_big' if margin_wickets >= 6 else 'B_small')
    else:
        targets.append(np.nan) # Ties/No Results

match_df['target'] = targets
match_df = match_df.dropna(subset=['target'])

label_encoder = LabelEncoder()
match_df['target_encoded'] = label_encoder.fit_transform(match_df['target'])

match_df.head() # Inspect the aggregated dataframe

,Match ID,Bat First,Bat Second,Venue,toss_winner,toss_decision,match_won_by,target,target_encoded
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,M Chinnaswamy Stadium,Royal Challengers Bengaluru,field,Kolkata Knight Riders,A_big,0
1,335983,Chennai Super Kings,Punjab Kings,Punjab Cricket Association Stadium,Chennai Super Kings,bat,Chennai Super Kings,A_big,0
2,335984,Rajasthan Royals,Delhi Capitals,Feroz Shah Kotla,Rajasthan Royals,bat,Delhi Capitals,B_big,2
3,335985,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai Indians,bat,Royal Challengers Bengaluru,B_small,3
4,335986,Deccan Chargers,Kolkata Knight Riders,Eden Gardens,Deccan Chargers,bat,Kolkata Knight Riders,B_small,3


Feature Engineering

In [15]:
features_list = []

def engineer_features(df, is_train=True):
    df_feats = df.copy()
    
    # Label Encode Categoricals
    cat_cols = ['Bat First', 'Bat Second', 'Venue', 'toss_decision']
    for col in cat_cols:
        if col in df_feats.columns:
            df_feats[col + '_enc'] = df_feats[col].astype('category').cat.codes
            if col + '_enc' not in features_list:
                features_list.append(col + '_enc')

    # Toss Winner logic
    if 'toss_winner' in df_feats.columns and 'Bat First' in df_feats.columns:
        df_feats['team_a_won_toss'] = (df_feats['toss_winner'] == df_feats['Bat First']).astype(int)
        if 'team_a_won_toss' not in features_list:
            features_list.append('team_a_won_toss')
            
    if is_train:
        return df_feats[features_list], df_feats['target_encoded']
    else:
        return df_feats[features_list]

X_train, y_train = engineer_features(match_df, is_train=True)
print(f"Features engineered: {features_list}")

Features engineered: ['Bat First_enc', 'Bat Second_enc', 'Venue_enc', 'toss_decision_enc', 'team_a_won_toss']


Optuna Hyperparameter Optimization

In [16]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 4,
        'eval_metric': 'mlogloss',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300)
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    log_losses = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params, random_state=42)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)
        loss = log_loss(y_va, preds, labels=[0, 1, 2, 3])
        log_losses.append(loss)
        
    return np.mean(log_losses)

# You can increase n_trials to 50 or 100 for better results if you have time
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

best_params = study.best_params
print(f"Best Log Loss found: {study.best_value:.4f}")
print(f"Best Parameters: {best_params}")

[I 2026-05-17 09:47:23,695] A new study created in memory with name: no-name-3ff71d7c-6904-45bd-a96a-4a3d0ef6c256
[I 2026-05-17 09:47:25,052] Trial 0 finished with value: 1.4013593734765526 and parameters: {'learning_rate': 0.02395969948179395, 'max_depth': 7, 'subsample': 0.825111824352375, 'colsample_bytree': 0.7455045526508182, 'n_estimators': 114}. Best is trial 0 with value: 1.4013593734765526.
[I 2026-05-17 09:47:26,584] Trial 1 finished with value: 1.3693488802521163 and parameters: {'learning_rate': 0.010732616928896332, 'max_depth': 5, 'subsample': 0.7573603462985466, 'colsample_bytree': 0.7489454128027659, 'n_estimators': 176}. Best is trial 1 with value: 1.3693488802521163.
[I 2026-05-17 09:47:27,633] Trial 2 finished with value: 1.398907416695345 and parameters: {'learning_rate': 0.04555150662843736, 'max_depth': 3, 'subsample': 0.6506681687220162, 'colsample_bytree': 0.7007986053002566, 'n_estimators': 179}. Best is trial 1 with value: 1.3693488802521163.
[I 2026-05-17 09:

Best Log Loss found: 1.3558
Best Parameters: {'learning_rate': 0.015752890567256917, 'max_depth': 3, 'subsample': 0.9414293152079922, 'colsample_bytree': 0.6705511205635227, 'n_estimators': 95}


Train Final Model

In [17]:
final_params = best_params.copy()
final_params.update({'objective': 'multi:softprob', 'num_class': 4})

final_model = xgb.XGBClassifier(**final_params, random_state=42)
final_model.fit(X_train, y_train)

print("Final model trained successfully.")

Final model trained successfully.


Prepare Test Set & Predict

In [ ]:

public_lb = clean_team_names(public_lb, ['team_a', 'team_b', 'toss_winner'])
schedule_df = clean_team_names(schedule, ['team_a', 'team_b'])

# 2. Dynamically determine Bat First / Bat Second for Public LB (2025)
def get_bat_first(row):
    if row['toss_decision'] == 'bat':
        return row['toss_winner']
    else: # If they chose to field, the OTHER team bats first
        return row['team_b'] if row['toss_winner'] == row['team_a'] else row['team_a']

def get_bat_second(row):
    if row['toss_decision'] == 'field':
        return row['toss_winner']
    else: # If they chose to bat, the OTHER team fields (bats second)
        return row['team_b'] if row['toss_winner'] == row['team_a'] else row['team_a']

public_lb['actual_bat_first'] = public_lb.apply(get_bat_first, axis=1)
public_lb['actual_bat_second'] = public_lb.apply(get_bat_second, axis=1)

# Format 2025 Holdout (Public LB)
public_lb_test = pd.DataFrame({
    'Match ID': public_lb['match_id'],
    'Bat First': public_lb['actual_bat_first'],   # FIXED: Now strictly Team A
    'Bat Second': public_lb['actual_bat_second'], # FIXED: Now strictly Team B
    'Venue': public_lb['venue'],
    'toss_decision': public_lb['toss_decision'],
    'toss_winner': public_lb['toss_winner']
})

private_lb_test = pd.DataFrame({
    'Match ID': schedule_df['match_id'],
    'Bat First': schedule_df['team_a'],  # Forces model to evaluate Home Team as A
    'Bat Second': schedule_df['team_b'], # Forces model to evaluate Away Team as B
    'Venue': schedule_df['venue'],
    'toss_decision': 'field',            # Baseline imputation
    'toss_winner': schedule_df['team_b'] # Baseline imputation
})

combined_test = pd.concat([public_lb_test, private_lb_test], ignore_index=True)
X_test = engineer_features(combined_test, is_train=False)

probas = final_model.predict_proba(X_test)
print("Predictions generated successfully!")

Predictions generated successfully!


Format Submission CSV

In [20]:
class_names = label_encoder.classes_

submission = pd.DataFrame({'match_id': combined_test['Match ID']})
for i, class_name in enumerate(class_names):
    submission[class_name] = probas[:, i]

# Ensure column order perfectly matches the sample submission
submission = submission[['match_id', 'A_small', 'A_big', 'B_small', 'B_big']]

# Save to CSV
submission.to_csv('submission.csv', index=False)

print("Pipeline complete! 'submission.csv' is ready for Kaggle.")
submission.head(53) # Verify the output looks correct!

Pipeline complete! 'submission.csv' is ready for Kaggle.


,match_id,A_small,A_big,B_small,B_big
0,1473488,0.189792,0.267112,0.202861,0.340234
1,1473489,0.192766,0.279055,0.208917,0.319262
2,1473490,0.276981,0.216643,0.196256,0.310120
3,1473491,0.224105,0.240322,0.180433,0.355140
4,1473492,0.197542,0.265176,0.221670,0.315612
5,1473493,0.206626,0.250967,0.185616,0.356791
6,1473494,0.228250,0.238154,0.199192,0.334404
7,1473495,0.229636,0.241561,0.192574,0.336229
8,1473497,0.228829,0.257785,0.194062,0.319324
9,1473498,0.266821,0.251567,0.191533,0.290079
